# الدرس 04 - نمط تصميم استخدام الأدوات

في هذا الدرس سوف تتعلم نمط تصميم **استخدام الأدوات** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل وكيل مايكروسوفت (Python). نغطي:

- تعريف أدوات الوظائف باستخدام مزخرف `@tool` والمعاملات المُحددة النوع
- توفير مخططات الأدوات حتى يعرف النموذج ما يفعله كل أداة
- التحكم في تنفيذ الأدوات باستخدام `approval_mode`
- إرجاع **مخرجات منظمة** عبر نماذج Pydantic و`response_format`

السيناريو هو **وكيل حجز سفر** يمكنه البحث عن الوجهات، التحقق من التوافر، واسترجاع معلومات الرحلات الجوية.


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## تعريف الأدوات باستخدام المزخرف @tool

يقوم المزخرف `@tool` بتحويل دالة بايثون عادية إلى أداة يمكن للوكيل استدعاؤها.  
نقاط رئيسية:

- تصبح **سلسلة التوثيق** وصف الأداة الذي يراه النموذج.  
- تُحدد **التعليقات التوضيحية للنوع** (بما في ذلك `Annotated` مع الأوصاف) مخطط الأداة.  
- يتحكم `approval_mode` فيما إذا كان يجب على المستخدم الموافقة على كل استدعاء قبل تنفيذه.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## إنشاء وكيل مع أدوات متعددة

مرر جميع الأدوات الثلاث إلى العميل حتى يتمكن النموذج من استدعاء أي منها يحتاجه للإجابة على سؤال المستخدم.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## الإخراج المهيكل مع الأدوات

عن طريق تعيين `response_format` إلى نموذج Pydantic، يُجبر الوكيل على إرجاع كائن JSON مضبوط النوع بدلاً من النص الحر. هذا مفيد عندما يحتاج الكود المتتالي إلى استهلاك النتيجة برمجياً.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## أنماط الموافقة على الأداة

تتحكم المعلمة `approval_mode` في `@tool` فيما إذا كانت مكالمات الأداة تتطلب موافقة بشرية قبل التنفيذ:

| الوضع | السلوك |
|---|---|
| `"never_require"` | تعمل الأداة تلقائيًا — لا حاجة لتأكيد المستخدم. |
| `"always_require"` | يجب الموافقة على كل مكالمة من قبل المستخدم قبل التنفيذ. |

استخدم `"always_require"` للأدوات التي لها تأثيرات جانبية (مثل حجز رحلة طيران، أو خصم من بطاقة ائتمان) بحيث يبقى البشر ضمن الحلقة.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## الملخص

في هذا الدرس تعلمت كيفية:

1. **تعريف الأدوات** باستخدام المزخرف `@tool` مع معلمات مكتوبة ونصوص توضيحية تعمل كمخطط للأداة.
2. **تجميع أدوات متعددة** حتى يتمكن الوكيل من استدعائها بالتتابع للإجابة على الاستفسارات المعقدة.
3. **إرجاع مخرجات منظمة** بتمرير نموذج Pydantic كـ `response_format`.
4. **التحكم في الموافقة على الأداة** باستخدام `approval_mode` للحفاظ على وجود عنصر بشري في العملية للعمليات الحساسة.

تشكل هذه الأنماط الأساس لبناء وكلاء موثوقين وجاهزين للإنتاج يمكنهم التفاعل بأمان مع الأنظمة الخارجية.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
